# Genetic Algorithm Experiment
#
Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis
#
Dataset: SST-2
Model: Qwen2.5-1.5B-Instruct
Optimization metric: Macro F1
#
The GA optimizes natural-language prompts, not model weights.
Only the optimization split is used during evolution.
Validation and final test sets are not used for fitness evaluation.

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import subprocess
import random
import time
import importlib.util
from pathlib import Path

REPO_URL = (
    "https://github.com/sarrazer24/"
    "evolutionary-prompt-optimization.git"
)

REPO_DIR = Path(
    "/kaggle/working/evolutionary-prompt-optimization"
)

# Clone the main branch or update an existing clone.
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository."
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
        check=True,
    )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "main"],
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "pull",
            "--ff-only",
            "origin",
            "main",
        ],
        check=True,
    )

os.chdir(REPO_DIR)

# Install only missing dependencies.
# IMPORTANT: Do not reinstall torch on Kaggle.
# Kaggle's preinstalled PyTorch is configured for its GPU environment.

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ],
        check=True,
    )
else:
    print("All required packages are already installed.")

# Make project modules importable.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project directory:", Path.cwd())
print("Python version:", sys.version)
print("Setup completed.")

## 2. Imports and Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.sst2_data import (
    load_saved_splits,
    stratified_sample,
)

from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface

from src.prompts.baseline_prompts import (
    get_baseline_prompt_texts,
)

from src.evaluation.evaluation import PredictionCache

from src.genetic_algorithm.optimizer import run_evolution


# Experiment seed.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

## 3. Check GPU Availability

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable GPU acceleration in Kaggle settings."
    )

GPU_NAME = torch.cuda.get_device_name(0)
CUDA_VERSION = torch.version.cuda

print("GPU:", GPU_NAME)
print("CUDA version:", CUDA_VERSION)
print("PyTorch version:", torch.__version__)

## 4. Load SST-2 Data
#
We use a fixed stratified sample from splits.opt.
The same examples are used to evaluate every candidate prompt.
#
The validation and official test sets are not used by the GA.

In [ ]:
DATA_DIR = Path("data/processed")

required_files = [
    DATA_DIR / "opt.csv",
    DATA_DIR / "val.csv",
    DATA_DIR / "split_info.json",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required dataset file not found: {file_path}"
        )

splits = load_saved_splits(str(DATA_DIR))

N_OPT = 1000

if len(splits.opt) < N_OPT:
    raise ValueError(
        f"Optimization split contains only {len(splits.opt)} "
        f"examples, but {N_OPT} were requested."
    )

opt_sample = stratified_sample(
    splits.opt,
    n=N_OPT,
    seed=SEED,
)

examples = opt_sample["text"].tolist()
labels = opt_sample["label"].tolist()

assert len(examples) == N_OPT
assert len(examples) == len(labels)
assert set(labels).issubset({0, 1})

print("Optimization sample size:", len(opt_sample))

print("\nClass distribution:")
print(opt_sample["label"].value_counts().sort_index())

print("\nExample:")
print(examples[0])
print("Label:", labels[0])

## 5. Configure and Load the LLM
#
ModelConfig provides the pinned model revision
and the project's existing inference configuration.

In [ ]:
model_config = ModelConfig()

print("Model:", model_config.model_id)
print("Revision:", model_config.revision)
print("Batch size:", model_config.batch_size)

llm = LLMInterface(model_config)
llm.load_model()
llm.set_seed(SEED)

print("\nModel loaded successfully.")
print("Model ID:", llm.model_id)
print("Inference configuration:", llm.inference_config)

## 6. Load Manual Baseline Prompts
#
These manually designed prompts seed the initial population.
The initialization module validates and deduplicates them,
then generates additional prompts to reach the population size.

In [ ]:
seed_prompts = get_baseline_prompt_texts()

if not seed_prompts:
    raise ValueError("No baseline prompts were returned.")

print("Number of manual baseline prompts:", len(seed_prompts))

for i, prompt in enumerate(seed_prompts, start=1):
    print(f"\n--- Baseline Prompt {i} ---")
    print(prompt)

## 7. Genetic Algorithm Configuration

In [ ]:
POPULATION_SIZE = 12
GENERATIONS = 10

ELITE_COUNT = 2
TOURNAMENT_SIZE = 3

CROSSOVER_RATE = 0.70
MUTATION_RATE = 0.30

EARLY_STOPPING_PATIENCE = None

print("Genetic Algorithm configuration:")
print("Population size:", POPULATION_SIZE)
print("Generations:", GENERATIONS)
print("Elite count:", ELITE_COUNT)
print("Tournament size:", TOURNAMENT_SIZE)
print("Crossover rate:", CROSSOVER_RATE)
print("Mutation rate:", MUTATION_RATE)
print("Early stopping:", EARLY_STOPPING_PATIENCE)

## 8. Initialize Prediction Cache and Output Directory
#
One shared cache is used throughout the experiment.
This avoids repeating inference for identical prompt-example pairs.

In [ ]:
prediction_cache = PredictionCache()

OUTPUT_DIR = Path(
    "/kaggle/working/outputs/"
    "genetic_algorithm/experiment_01"
)

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output directory is not empty: {OUTPUT_DIR}. "
        "Choose a new experiment directory to avoid overwriting results."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Prediction cache initialized.")
print("Output directory:", OUTPUT_DIR)

## 9. Run the Genetic Algorithm
#
The GA performs the following steps:
#
1. Initialize the prompt population.
2. Evaluate each prompt using Macro F1.
3. Select parents using tournament selection.
4. Preserve elite individuals.
5. Apply crossover and mutation.
6. Repeat for the configured number of generations.
#
The final held-out test set is not used.

In [ ]:
print("Starting Genetic Algorithm experiment...")

experiment_start_time = time.perf_counter()

result = run_evolution(
    llm=llm,
    examples=examples,
    labels=labels,

    population_size=POPULATION_SIZE,
    generations=GENERATIONS,

    elite_count=ELITE_COUNT,
    tournament_size=TOURNAMENT_SIZE,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,

    seed_prompts=seed_prompts,
    seed=SEED,

    cache=prediction_cache,
    batch_size=None,

    early_stopping_patience=EARLY_STOPPING_PATIENCE,

    output_dir=str(OUTPUT_DIR),
)

experiment_duration = time.perf_counter() - experiment_start_time

print("\nEvolution completed.")
print(f"Execution time: {experiment_duration:.2f} seconds")

## 10. Inspect the Best Evolved Prompt

In [ ]:
best_individual = result["best_individual"]

print("=" * 70)
print("BEST EVOLVED PROMPT")
print("=" * 70)

print(best_individual.prompt)

print("\nBest individual information:")
print("Individual ID:", best_individual.id)
print("Generation:", best_individual.generation)
print("Macro F1:", best_individual.fitness)
print("Accuracy:", best_individual.accuracy)
print("Parent IDs:", best_individual.parent_ids)

print("\nEvaluation metadata:")
print(
    json.dumps(
        best_individual.metadata,
        indent=2,
        default=str,
    )
)

## 11. Analyze Evolution History
#
The history contains the best, average and worst fitness
for each evaluated generation.

In [ ]:
history_df = pd.DataFrame(result["history"])

history_columns = [
    "generation",
    "best_fitness",
    "average_fitness",
    "worst_fitness",
    "diversity",
    "total_invalid",
    "total_model_texts",
]

print(history_df[history_columns].to_string(index=False))

# Note:
# total_model_texts represents uncached example texts evaluated.
# It is not the number of physical GPU model-generation calls.

## 12. Plot Fitness Evolution

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history_df["generation"],
    history_df["best_fitness"],
    marker="o",
    label="Best Macro F1",
)

plt.plot(
    history_df["generation"],
    history_df["average_fitness"],
    marker="s",
    label="Average Macro F1",
)

plt.plot(
    history_df["generation"],
    history_df["worst_fitness"],
    marker="^",
    label="Worst Macro F1",
)

plt.xlabel("Generation")
plt.ylabel("Macro F1")
plt.title("Genetic Algorithm Fitness Evolution")

plt.xticks(history_df["generation"])
plt.grid(True)
plt.legend()
plt.tight_layout()

fitness_plot_path = OUTPUT_DIR / "fitness_evolution.png"
plt.savefig(fitness_plot_path, dpi=300)
plt.show()

print("Fitness plot saved to:", fitness_plot_path)

## 13. Plot Population Diversity
#
Diversity is measured using exact prompt-string uniqueness.
It is not a semantic similarity measure.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["generation"],
    history_df["diversity"],
    marker="o",
)

plt.xlabel("Generation")
plt.ylabel("Exact Prompt Diversity")
plt.title("Population Diversity Across Generations")

plt.xticks(history_df["generation"])
plt.ylim(0, 1.05)
plt.grid(True)
plt.tight_layout()

diversity_plot_path = OUTPUT_DIR / "population_diversity.png"
plt.savefig(diversity_plot_path, dpi=300)
plt.show()

print("Diversity plot saved to:", diversity_plot_path)

## 14. Save Experiment Configuration

In [ ]:
experiment_config = {
    "experiment_name": "genetic_algorithm_experiment_01",

    "dataset": "SST-2",
    "optimization_split": "opt",
    "optimization_sample_size": N_OPT,
    "sampling_seed": SEED,

    "model_id": llm.model_id,
    "model_revision": model_config.revision,
    "inference_config": llm.inference_config,

    "gpu_name": GPU_NAME,
    "cuda_version": CUDA_VERSION,
    "pytorch_version": torch.__version__,

    "population_size": POPULATION_SIZE,
    "generations_requested": GENERATIONS,
    "generations_evaluated": len(history_df),

    "elite_count": ELITE_COUNT,
    "tournament_size": TOURNAMENT_SIZE,

    "crossover_rate": CROSSOVER_RATE,
    "mutation_rate": MUTATION_RATE,

    "early_stopping_patience": EARLY_STOPPING_PATIENCE,

    "fitness_metric": "Macro F1",
    "number_of_manual_seed_prompts": len(seed_prompts),

    "random_seed": SEED,
    "execution_time_seconds": round(experiment_duration, 2),
}

config_path = OUTPUT_DIR / "experiment_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        experiment_config,
        file,
        indent=4,
        default=str,
    )

print("Experiment configuration saved:", config_path)

## 15. Final Experiment Summary

In [ ]:
print("=" * 70)
print("GENETIC ALGORITHM EXPERIMENT SUMMARY")
print("=" * 70)

print("Model:", llm.model_id)
print("Dataset: SST-2")
print("Optimization examples:", N_OPT)

print("Generations evaluated:", len(history_df))
print("Population size:", POPULATION_SIZE)

print("Best Macro F1:", round(best_individual.fitness, 4))
print("Best accuracy:", round(best_individual.accuracy, 4))

print("Best prompt generation:", best_individual.generation)
print(f"Execution time: {experiment_duration:.2f} seconds")

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

print("\nExperiment completed successfully.")